# LOW thinking: grocery replenishment

Rerun of the original dataset and evaluator with explicit LOW thinking. Usage records distinguish prompt, final-output, thinking and total tokens. Missing fields stay null. Tool creation is measured separately from task execution. The optional comparison later uses fresh MEDIUM calls and an explicitly requested Gemini-written tool; that arm is an ablation, not autonomous self-improvement.


## Recorded results — October 6, 2026

| Held-out arm | Passed / attempts | Tool calls | Input | Output | Reported thinking* | Total | Usage coverage |
|---|---:|---:|---:|---:|---:|---:|---:|
| LOW_no_tool | 8/8 | 0 | 10,226 | 2,694 | 281 | 13,201 | 8/8 |
| MEDIUM_no_tool | 8/8 | 0 | 10,226 | 2,289 | 14,077 | 26,592 | 8/8 |
| LOW_tool_available | 8/8 | 8 | 11,074 | 3,441 | 707 | 15,222 | 8/8 |
| MEDIUM_tool_available | 5/8 | 2 | 6,931 | 1,801 | 12,989 | 21,721 | 5/8 |

*Missing thinking-count fields remain null in the raw record. The table sums only reported counts. For all completed responses here, total minus input minus output reconciles with reported thinking (or zero when the field is omitted). LOW is not a fixed token budget. MEDIUM request failures have no returned usage; their totals are incomplete and cannot be compared as equal-work cost totals.

LOW exposed one real development subtotal/savings error (USD 0.70), plus two exact-status rubric failures where asking to verify an ambiguous oats label was arguably reasonable. Keep those separate when interpreting the 5/8 baseline score. The instruction/memory proposal passed development 8/8 but held-out 6/8: it stopped requesting approval when substitution preferences were missing. Its claim that checklist omissions were penalties was inaccurate; checklists were diagnostic only.

The Gemini-written calculator was explicitly requested for a controlled comparison, rather than chosen autonomously. It passed 7/8 development and 8/8 held-out checks; the remaining development failure was recommendation status, not arithmetic. LOW tool use cost **15,222 vs 13,201** held-out tokens (**15.3% more**), plus **4,270** creation tokens. It was not retained as a token-efficiency improvement. MEDIUM tool-available had three read timeouts and usage for only five attempts. A calculator does not settle uncertain labels, approval or local purchasability.

The complete notebook attempted **66 calls** and returned **148,810 reported tokens**, including proposals, development and all comparison arms. Repeated attempts are not independent new tasks. The fixed improver was never modified; this is not recursive improvement.

Configuration: `gemini-3.8-flash`, `google-genai==2.28.0`, explicit LOW/MEDIUM, temperature 0, output cap 8,192, request timeout 90 seconds, no retries. The output cap includes thinking. These data test LOW versus MEDIUM; they do not test HIGH. Sources: [Google thinking configuration](https://ai.google.dev/gemini-api/docs/thinking), [SDK types and usage fields](https://googleapis.github.io/python-genai/genai.html#genai.types.GenerateContentResponseUsageMetadata).


In [ ]:
%pip -q install google-genai==2.28.0

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## 1. Freeze the evidence, not the world
These are short factual summaries of retailer pages, with source URLs. The task
agent reads these text records; it does not receive the evaluator's normalized
product table. The two are shown together here so a reader can audit the exercise.
Every listing's generic stock label still requires checking a chosen store.
We deliberately don't invent fees or interpret incomplete promotion terms.

In [4]:
import os, re, json, time, statistics
from decimal import Decimal
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types
from IPython.display import JSON, display

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"], vertexai=False,
    http_options=types.HttpOptions(timeout=90000,
        retry_options=types.HttpRetryOptions(attempts=1)))
THINKING_LEVEL = "LOW"
usage, calls = [], []

def ask(prompt, schema):
    start = time.monotonic()
    call = {"attempt":len(calls)+1, "thinking_level":THINKING_LEVEL}
    calls.append(call)
    try:
        response = client.models.generate_content(model=MODEL, contents=prompt,
            config={"temperature":0, "max_output_tokens":8192,
                    "thinking_config":{"thinking_level":THINKING_LEVEL},
                    "response_mime_type":"application/json", "response_json_schema":schema,
                    "automatic_function_calling":{"disable":True}})
        u = response.usage_metadata
        resources = {"tokens":u.total_token_count, "prompt_tokens":u.prompt_token_count,
            "output_tokens":u.candidates_token_count, "thinking_tokens":u.thoughts_token_count,
            "cached_tokens":u.cached_content_token_count,
            "seconds":round(time.monotonic()-start, 3)}
        # Preserve missing metadata as null, not zero.
        usage.append(resources)
        reason = str(response.candidates[0].finish_reason) if response.candidates else "no candidate"
        call.update(resources, finish_reason=reason)
        if not response.candidates or response.candidates[0].finish_reason != "STOP":
            error = RuntimeError("Incomplete model response: " + reason)
            error.resources = resources
            raise error
        return json.loads(response.text), resources
    except Exception as error:
        call.update(error_type=type(error).__name__, status=getattr(error,"code",None),
                    seconds=round(time.monotonic()-start,3))
        raise

LISTINGS = [{"id":"q42","url":"https://www.target.com/p/-/A-13331320","retrieved_on":"2026-10-06","evidence":"Quaker Old Fashioned Oats; selected package 42 oz. Displayed item price USD 5.49. Generic page stock label: in stock. Local fulfillment not established. An oatmeal promotion is displayed; complete eligibility and checkout application were not captured."},{"id":"g42","url":"https://www.target.com/p/-/A-79364999","retrieved_on":"2026-10-06","evidence":"Good & Gather Old Fashioned Oats; selected package 42 oz. Displayed item price USD 4.39. Generic page stock label: in stock. Local fulfillment not established. An oatmeal promotion is displayed; complete eligibility and checkout application were not captured. The old-fashioned title conflicts with a highlight that describes quick oats; verify the selected product label if this matters."},{"id":"organic18","url":"https://www.target.com/p/-/A-79364857","retrieved_on":"2026-10-06","evidence":"Good & Gather Organic Old Fashioned Oats; selected package 18 oz. Displayed item price USD 3.89. Generic page stock label: in stock. Local fulfillment not established. An oatmeal promotion is displayed; complete eligibility and checkout application were not captured. The old-fashioned title conflicts with a highlight that describes quick oats; verify the selected product label if this matters."},{"id":"quick42","url":"https://www.target.com/p/-/A-79365223","retrieved_on":"2026-10-06","evidence":"Good & Gather Quick Oats; selected package 42 oz. Displayed item price USD 4.39. Generic page stock label: in stock. Local fulfillment not established. An oatmeal promotion is displayed; complete eligibility and checkout application were not captured."},{"id":"b16","url":"https://www.target.com/p/-/A-13156213","retrieved_on":"2026-10-06","evidence":"Barilla Spaghetti; selected package 16 oz. Displayed item price USD 1.89. Generic page stock label: in stock. Local fulfillment not established."},{"id":"g16","url":"https://www.target.com/p/-/A-89301598","retrieved_on":"2026-10-06","evidence":"Good & Gather Spaghetti; selected package 16 oz. Displayed item price USD 1.29. Generic page stock label: in stock. Local fulfillment not established. The page offers a separate 16 oz / 32 oz size selector."},{"id":"g32","url":"https://www.target.com/p/-/A-78779119","retrieved_on":"2026-10-06","evidence":"Good & Gather Spaghetti; selected package 32 oz. Displayed item price USD 1.99. Generic page stock label: in stock. Local fulfillment not established. The page offers a separate 16 oz / 32 oz size selector."},{"id":"j16","url":"https://www.target.com/p/-/A-13292010","retrieved_on":"2026-10-06","evidence":"Jif Creamy Peanut Butter; selected package 16 oz. Displayed item price USD 2.99. Generic page stock label: in stock. Local fulfillment not established. Previous displayed price was USD 3.19; current displayed price is USD 2.99."},{"id":"pb16","url":"https://www.target.com/p/-/A-84067786","retrieved_on":"2026-10-06","evidence":"Good & Gather Creamy Peanut Butter; selected package 16 oz. Displayed item price USD 1.99. Generic page stock label: in stock. Local fulfillment not established."},{"id":"pb40","url":"https://www.target.com/p/-/A-84067787","retrieved_on":"2026-10-06","evidence":"Good & Gather Creamy Peanut Butter; selected package 40 oz. Displayed item price USD 3.59. Generic page stock label: in stock. Local fulfillment not established. Previous displayed price was USD 3.99; current displayed price is USD 3.59."}]

REFERENCE = {'q42': {'family': 'oats', 'oz': 42, 'price': '5.49', 'variant': 'old_fashioned', 'organic': False, 'url': 'https://www.target.com/p/-/A-13331320'}, 'g42': {'family': 'oats', 'oz': 42, 'price': '4.39', 'variant': 'old_fashioned', 'organic': False, 'url': 'https://www.target.com/p/-/A-79364999'}, 'organic18': {'family': 'oats', 'oz': 18, 'price': '3.89', 'variant': 'old_fashioned', 'organic': True, 'url': 'https://www.target.com/p/-/A-79364857'}, 'quick42': {'family': 'oats', 'oz': 42, 'price': '4.39', 'variant': 'quick', 'organic': False, 'url': 'https://www.target.com/p/-/A-79365223'}, 'b16': {'family': 'pasta', 'oz': 16, 'price': '1.89', 'variant': 'spaghetti', 'organic': False, 'url': 'https://www.target.com/p/-/A-13156213'}, 'g16': {'family': 'pasta', 'oz': 16, 'price': '1.29', 'variant': 'spaghetti', 'organic': False, 'url': 'https://www.target.com/p/-/A-89301598'}, 'g32': {'family': 'pasta', 'oz': 32, 'price': '1.99', 'variant': 'spaghetti', 'organic': False, 'url': 'https://www.target.com/p/-/A-78779119'}, 'j16': {'family': 'peanut_butter', 'oz': 16, 'price': '2.99', 'variant': 'creamy', 'organic': False, 'url': 'https://www.target.com/p/-/A-13292010'}, 'pb16': {'family': 'peanut_butter', 'oz': 16, 'price': '1.99', 'variant': 'creamy', 'organic': False, 'url': 'https://www.target.com/p/-/A-84067786'}, 'pb40': {'family': 'peanut_butter', 'oz': 40, 'price': '3.59', 'variant': 'creamy', 'organic': False, 'url': 'https://www.target.com/p/-/A-84067787'}}
print("Model:", MODEL, "| Captured listing pages:", len(LISTINGS))
display(JSON(LISTINGS, expanded=False))

Model: gemini-3.8-flash | Captured listing pages: 10


Expand to inspect records.

## 2. Shopping occasions and a limited evaluator
Four requests are used for development; four are held out. Each gets two attempts.
These are different example occasions, not independent market snapshots.
The request describes the usual products, quantities, and substitution preferences.
An unspecified preference should lead to a conditional recommendation or question.

We check product IDs, source links, whole packs, sufficient quantity, stated
preferences, displayed-price arithmetic, and acknowledgment of missing checkout
information. **Any valid cheaper candidate can succeed; there is no optimum
answer or exact target savings.** Keeping the usual basket or asking for approval
can be appropriate. This rubric is a partial proxy, not a measure of the complete
shopping experience. It doesn't grade taste or prove that a suggested basket can
be purchased locally. The evaluator's facts and expectations aren't sent to the agent.
Uncertainty checklist completeness is a separate diagnostic, not an exact-word
pass/fail test. Equivalent source URLs for the same Target product are accepted.
The structured rubric still cannot replace a shopper reading the explanation.


In [6]:
def occasion(id, usual, preferences, allowed, status, max_pack=None):
    return {"id":id, "usual":usual, "preferences":preferences,
            "allowed":allowed, "expected_status":status, "max_pack":max_pack or {}}

OPEN = ["q42","g42","organic18","b16","g16","g32","j16","pb16","pb40"]
development = [
    occasion("weekly", {"q42":1,"b16":2,"j16":1},
        "Store brands are okay for old-fashioned oats and regular spaghetti. Keep Jif creamy peanut butter. No quick oats.",
        ["q42","g42","organic18","b16","g16","g32","j16"], "cheaper_candidate"),
    occasion("stock-up", {"q42":2,"b16":3,"j16":2},
        "Store brands are approved for all three products. Only old-fashioned oats, regular spaghetti and creamy peanut butter. Buy enough whole packs; larger packs are fine.",
        OPEN, "cheaper_candidate"),
    occasion("small-cupboard", {"j16":2},
        "Store-brand creamy peanut butter is okay, but jars must fit the cupboard: no jar larger than 16 oz.",
        ["j16","pb16"], "cheaper_candidate", {"peanut_butter":16}),
    occasion("brand-loyal", {"q42":1,"b16":1,"j16":1},
        "Keep these exact brands, varieties and package sizes. Compare captured ordinary item prices; don't assume promotion eligibility.",
        ["q42","b16","j16"], "no_cheaper_found"),
]
held_out = [
    occasion("family-refill", {"q42":1,"b16":4,"j16":2},
        "Private labels are approved for rolled old-fashioned oats and regular spaghetti; peanut butter must stay Jif creamy. Larger pasta packs are okay.",
        ["q42","g42","organic18","b16","g16","g32","j16"], "cheaper_candidate"),
    occasion("organic-oats", {"organic18":2},
        "Oats must remain organic and old-fashioned. Do not trade that requirement for a lower unit price.",
        ["organic18"], "no_cheaper_found"),
    occasion("pb-replenishment", {"j16":3},
        "Store-brand creamy peanut butter is approved. Large jars are fine, but compare the actual cost of enough whole jars, not only unit prices.",
        ["j16","pb16","pb40"], "cheaper_candidate"),
    occasion("preferences-missing", {"q42":1,"b16":2,"j16":1},
        "Find a cheaper refill. I haven't said whether I accept different brands or varieties; don't label an unapproved substitution acceptable.",
        OPEN, "needs_approval"),
]

LINE = {"type":"object", "properties":{"listing_id":{"type":"string"},
    "quantity":{"type":"integer"}, "source_url":{"type":"string"}},
    "required":["listing_id","quantity","source_url"], "additionalProperties":False}
ANSWER = {"type":"object", "properties":{
    "status":{"type":"string", "enum":["cheaper_candidate","no_cheaper_found","needs_approval"]},
    "lines":{"type":"array", "items":LINE},
    "estimated_item_subtotal":{"type":"string"}, "usual_item_subtotal":{"type":"string"},
    "estimated_item_savings":{"type":"string"}, "checkout_verified":{"type":"boolean"},
    "checks_needed":{"type":"array", "items":{"type":"string", "enum":["local_stock","current_price","fees","promotion_terms","product_label"]}},
    "questions":{"type":"array", "items":{"type":"string"}}, "explanation":{"type":"string"}},
    "required":["status","lines","estimated_item_subtotal","usual_item_subtotal",
                "estimated_item_savings","checkout_verified","checks_needed","questions","explanation"],
    "additionalProperties":False}
BASELINE = {"instruction":"Help the shopper find a cheaper acceptable refill. Use the supplied listing evidence, respect their preferences, cite chosen product sources, and be honest about what remains unknown.", "memory":""}

def item_total(lines):
    return sum((Decimal(REFERENCE[k]["price"])*q for k,q in lines.items()), Decimal("0"))

def money(text):
    text = str(text).strip()
    if not re.fullmatch(r"(?:USD\s*|\$)?-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?", text):
        raise ValueError("Unparseable amount")
    return Decimal(text.removeprefix("USD").strip().removeprefix("$").replace(",", ""))

def grade(case, answer):
    errors, picked, supplied, needs = [], {}, {}, {}
    for k,q in case["usual"].items():
        family = REFERENCE[k]["family"]
        needs[family] = needs.get(family,0) + REFERENCE[k]["oz"]*q
    for line in answer["lines"]:
        k,q = line["listing_id"],line["quantity"]
        if k not in REFERENCE or k in picked or type(q) is not int or q<1:
            errors.append("Invalid listing, duplicate line, or nonpositive quantity"); continue
        picked[k] = q
        r = REFERENCE[k]; family = r["family"]
        from urllib.parse import urlparse
        parsed = urlparse(line["source_url"])
        product_id = r["url"].split("A-")[-1]
        if parsed.hostname not in ("target.com","www.target.com") or not re.search(r"/A-"+product_id+r"(?:/|$)", parsed.path):
            errors.append("Wrong product source")
        if k not in case["allowed"]: errors.append("Unapproved brand or variety")
        if r["oz"] > case["max_pack"].get(family,100000): errors.append("Pack exceeds storage limit")
        supplied[family] = supplied.get(family,0) + r["oz"]*q
    if set(supplied) != set(needs) or any(supplied.get(f,0)<n for f,n in needs.items()):
        errors.append("Basket doesn't replenish the requested quantities")
    for family,n in supplied.items():
        smallest = min((REFERENCE[k]["oz"] for k in picked if REFERENCE[k]["family"]==family), default=0)
        if n-smallest >= needs.get(family,0): errors.append("Unnecessary extra whole pack")
    original, candidate = item_total(case["usual"]),item_total(picked)
    try:
        if money(answer["usual_item_subtotal"]) != original: errors.append("Wrong usual subtotal")
        if money(answer["estimated_item_subtotal"]) != candidate: errors.append("Wrong candidate subtotal")
        if money(answer["estimated_item_savings"]) != original-candidate: errors.append("Wrong item savings")
    except Exception: errors.append("Unparseable money amount")
    if answer["status"] != case["expected_status"]: errors.append("Wrong recommendation/approval status")
    if answer["status"] == "cheaper_candidate" and candidate>=original: errors.append("No item-price saving")
    if answer["status"] == "no_cheaper_found" and picked!=case["usual"]: errors.append("Keep the usual basket when no cheaper option is supported")
    if answer["status"] == "needs_approval" and not answer["questions"]: errors.append("Missing substitution question")
    if answer["checkout_verified"]: errors.append("Checkout was not verified")
    # Diagnostics, not exact-word pass/fail tests: a human reads the explanation.
    missing_checks = sorted({"local_stock","current_price","fees"}-set(answer["checks_needed"]))
    unresolved_label = any(k in picked for k in ("g42","organic18")) and "product_label" not in answer["checks_needed"]
    return {"appropriate":not errors, "errors":errors,
        "item_savings":str(original-candidate), "estimated_only":not answer["checkout_verified"],
        "uncertainty_checklist_missing":missing_checks, "label_check_not_listed":unresolved_label}

## 3. Real responses first, then one improvement proposal
The task agent gets a request and listing evidence, not a hand-labeled diagnosis.
The fixed improver sees development responses and evaluator feedback only. It
can retain the policy, revise instructions, or add procedural memory. We don't
force a failure or tell it which repair to choose. The saved trajectories also
let a human judge whether the explanations are useful beyond the scored fields.

In [8]:
REPEATS = 2

def run(job):
    artifact,case,repeat = job
    prompt = (artifact["instruction"] + "\nProcedural memory: " + artifact["memory"] +
        "\nTask: propose a cheaper acceptable refill or explain the next step. "
        "Use the captured ordinary displayed item prices, before unverified promotions, tax or fees. "
        "Return a full proposed basket (or the usual basket if keeping it), "
        "bare decimal dollar strings, source URLs, and a short explanation. "
        "A proposed basket is not a verified checkout.\nUsual basket by listing ID: " +
        json.dumps(case["usual"]) + "\nShopper preferences: " + case["preferences"] +
        "\nRetailer evidence: " + json.dumps(LISTINGS))
    row = {"occasion":case["id"],"repeat":repeat,"request":case["preferences"]}
    try:
        row["answer"],resources = ask(prompt, ANSWER)
        row.update(resources); row["feedback"] = grade(case,row["answer"])
    except Exception as error:
        row.update(getattr(error,"resources",{}))
        row["request_error"] = type(error).__name__
        row["request_status"] = getattr(error,"code",None)
        row["error_detail"] = str(error) if type(error) is RuntimeError else None
        row["feedback"] = {"appropriate":False,"errors":["API or execution error; inspect separately"]}
    return row

def evaluate(artifact,cases):
    with ThreadPoolExecutor(max_workers=2) as pool:
        return list(pool.map(run, [(artifact,c,r) for c in cases for r in range(REPEATS)]))

def summary(rows):
    return {"appropriate":sum(r["feedback"]["appropriate"] for r in rows),
        "attempts":len(rows), "occasions":len({r["occasion"] for r in rows}),
        "failed_requests":sum("request_error" in r for r in rows),
        "tokens":sum((r.get("tokens") or 0) for r in rows),
        "incomplete_checklists":sum(bool(r["feedback"].get("uncertainty_checklist_missing")) for r in rows)}

before_dev = evaluate(BASELINE, development)
print("Original development:",summary(before_dev))
for row in before_dev:
    if not row["feedback"]["appropriate"]: print(json.dumps(row))
CHANGE = {"type":"object", "properties":{
    "change":{"type":"string", "enum":["no_change","instruction","memory","both"]},
    "instruction":{"type":"string"},"memory":{"type":"string"},"reason":{"type":"string"}},
    "required":["change","instruction","memory","reason"],"additionalProperties":False}
IMPROVER = """Improve the task agent from these actual development trajectories.
You stay fixed; you cannot edit yourself, the model, evaluator, sources or harness.
Choose no change or revise the task instruction / procedural memory. Don't invent
failures, memorize case-specific quantities or prices, or generalize one shopper's
preferences to another. Keep instruction and memory each under 150 words. Propose
general lessons grounded in evidence. Success is a useful acceptable recommendation
with justified arithmetic and honest uncertainty, not globally optimal shopping.
If there is no measured weakness, no_change is valid. Return the existing artifact
unchanged for no_change. Explain the choice in under 100 words."""
proposal, proposal_usage = ask(IMPROVER + "\nCurrent agent: " + json.dumps(BASELINE) +
    "\nDevelopment trajectories: " + json.dumps(before_dev), CHANGE)
print("Proposal:",json.dumps(proposal,indent=2))
if proposal["change"] == "no_change":
    candidate = BASELINE
else:
    candidate = {k:proposal[k] for k in ("instruction","memory")}
after_dev = evaluate(candidate, development)
print("Candidate development:",summary(after_dev))
retained = (candidate if not any("request_error" in r for r in before_dev+after_dev)
    and summary(after_dev)["appropriate"]>summary(before_dev)["appropriate"] else BASELINE)
print("Retained:","candidate" if retained is candidate and candidate is not BASELINE else "original")

Original development: {'appropriate': 5, 'attempts': 8, 'occasions': 4, 'failed_requests': 0, 'tokens': 12995, 'incomplete_checklists': 6}
{"occasion": "weekly", "repeat": 0, "request": "Store brands are okay for old-fashioned oats and regular spaghetti. Keep Jif creamy peanut butter. No quick oats.", "answer": {"status": "needs_approval", "lines": [{"listing_id": "g42", "quantity": 1, "source_url": "https://www.target.com/p/-/A-79364999"}, {"listing_id": "g32", "quantity": 1, "source_url": "https://www.target.com/p/-/A-78779119"}, {"listing_id": "j16", "quantity": 1, "source_url": "https://www.target.com/p/-/A-13292010"}], "estimated_item_subtotal": "9.37", "usual_item_subtotal": "12.26", "estimated_item_savings": "2.89", "checkout_verified": false, "checks_needed": ["local_stock", "current_price", "product_label"], "questions": ["The Good & Gather 42 oz Old Fashioned Oats listing mentions quick oats in its highlights; would you like to verify the label or proceed with it?", "Would yo

## 4. New occasions, and human review
Held-out requests do not select or revise the candidate. We report both frozen
versions, including rejected proposals, and retain only a strict development
rubric gain with no API failures. Ties keep the original. If the proposal is
`no_change`, the two versions are independent repeats of the same policy.

Review at least one successful and one unsuccessful explanation when available.
A correct structured answer may still be unhelpful. These scores are preliminary
until reviewed by a shopper; they do not prove real purchases or economic benefit.

In [10]:
before_test,after_test = evaluate(BASELINE,held_out),evaluate(candidate,held_out)
print("Original held-out:",summary(before_test))
print("Candidate held-out:",summary(after_test))
for label,rows in (("original",before_test),("candidate",after_test)):
    for row in rows:
        print(label,row["occasion"],row["repeat"],row["feedback"])
print("Model calls attempted:",len(calls),"| Reported tokens:",sum(u["tokens"] for u in usage))
experiment = {"model":MODEL,"captured_on":"2026-10-06","proposal":proposal,
    "retained": "candidate" if retained is candidate and candidate is not BASELINE else "original",
    "development":{"original":before_dev,"candidate":after_dev},
    "held_out":{"original":before_test,"candidate":after_test},"calls_attempted":len(calls),"usage":usage}
display(JSON(experiment, expanded=False))

Original held-out: {'appropriate': 8, 'attempts': 8, 'occasions': 4, 'failed_requests': 0, 'tokens': 13201, 'incomplete_checklists': 6}
Candidate held-out: {'appropriate': 6, 'attempts': 8, 'occasions': 4, 'failed_requests': 0, 'tokens': 14602, 'incomplete_checklists': 0}
original family-refill 0 {'appropriate': True, 'errors': [], 'item_savings': '4.68', 'estimated_only': True, 'uncertainty_checklist_missing': ['fees'], 'label_check_not_listed': False}
original family-refill 1 {'appropriate': True, 'errors': [], 'item_savings': '4.68', 'estimated_only': True, 'uncertainty_checklist_missing': ['fees'], 'label_check_not_listed': False}
original organic-oats 0 {'appropriate': True, 'errors': [], 'item_savings': '0.00', 'estimated_only': True, 'uncertainty_checklist_missing': ['current_price', 'fees'], 'label_check_not_listed': False}
original organic-oats 1 {'appropriate': True, 'errors': [], 'item_savings': '0.00', 'estimated_only': True, 'uncertainty_checklist_missing': ['current_price

## Interpretation

This rerun preserves the original instruction/memory-only improvement experiment. The additional tool comparison below expands the editable space to a reusable arithmetic helper. The evaluator still checks shopping preferences independently; a calculator cannot resolve product ambiguity or missing permission.


In [12]:
experiment.update(thinking_level=THINKING_LEVEL, calls=calls)


## 5. Thinking versus executing a Gemini-written tool

This is a controlled 2×2 comparison: LOW/MEDIUM thinking and tool unavailable/available. The tool is explicitly requested from Gemini using development evidence. Its creation cost is separate; offering a tool does not force its use. Task instructions remain the original baseline, isolating the tool intervention. A simple JSON action protocol executes a call locally and returns its result directly; no second LLM synthesis call is charged. Real multi-turn agents may have additional overhead.

The efficiency candidate is selected on development data: no request failures, no accuracy regression, lower total tokens. Held-out tasks test it, never select it. This small sample cannot establish non-inferiority. For missing thought counts, `unattributed_tokens` reports total minus prompt minus output, rather than pretending the API reported zero.


In [13]:
# Explicit tool-writing ablation; the original instruction/memory proposal is above.
import ast
from decimal import ROUND_HALF_UP
BILL_HELPER = None

def compile_totals(source):
    tree = ast.parse(source)
    allowed = (ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Assign,
        ast.Return, ast.Expr, ast.Name, ast.Load, ast.Store, ast.Constant,
        ast.BinOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Call, ast.Attribute,
        ast.keyword, ast.Subscript, ast.Tuple, ast.List, ast.Dict,
        ast.GeneratorExp, ast.comprehension)
    if len(source)>2500 or len(list(ast.walk(tree)))>220 or len(tree.body)!=1:
        raise ValueError('Tool exceeds small arithmetic grammar')
    fn=tree.body[0]
    if not isinstance(fn,ast.FunctionDef) or fn.name!='basket_totals' or fn.decorator_list:
        raise ValueError('Expected basket_totals function')
    if fn.args.defaults or fn.args.kw_defaults or fn.args.vararg or fn.args.kwarg:
        raise ValueError('No defaults or variadic arguments')
    for node in ast.walk(tree):
        if not isinstance(node,allowed):raise ValueError('Disallowed syntax: '+type(node).__name__)
        if isinstance(node,(ast.Name,ast.arg)) and '__' in getattr(node,'id',getattr(node,'arg','')):
            raise ValueError('Disallowed name')
        if isinstance(node,ast.Attribute) and node.attr!='quantize':raise ValueError('Disallowed attribute')
        if isinstance(node,ast.Call):
            if isinstance(node.func,ast.Name) and node.func.id not in ('sum','Decimal','str','int'):
                raise ValueError('Disallowed call')
            if not isinstance(node.func,(ast.Name,ast.Attribute)):raise ValueError('Disallowed call target')
        # Generators may only iterate the bounded input lists, never recurse.
        if isinstance(node,ast.comprehension) and (node.is_async or node.ifs or
                not isinstance(node.iter,ast.Name) or node.iter.id not in ('lines','usual')):
            raise ValueError('Only bounded input generators permitted')
        if isinstance(node,ast.Constant) and len(str(node.value))>100:raise ValueError('Large constant')
    scope={'__builtins__':{},'Decimal':Decimal,'str':str,'int':int,'sum':sum,'ROUND_HALF_UP':ROUND_HALF_UP}
    exec(compile(tree,'generated_totals_tool','exec'),scope)
    return scope['basket_totals']

TOOL_CHANGE = {'type':'object','properties':{'tool_code':{'type':'string'},'reason':{'type':'string'}},
               'required':['tool_code','reason'],'additionalProperties':False}
tool_proposal, creation_usage = ask('''Write a reusable arithmetic helper for a shopping task agent.
This controlled ablation requests a tool even if existing baskets are correct,
to test whether execution reduces thinking tokens. Do not claim unseen failures.
Write exactly one function basket_totals(lines, usual, prices). lines and usual
are bounded lists of (listing_id, positive_integer_quantity) pairs; prices maps
listing IDs to decimal dollar strings parsed by the TASK agent from evidence.
Return a dict with string-valued estimated_item_subtotal, usual_item_subtotal,
estimated_item_savings. Use Decimal with full precision and final cents. No
imports, annotations, defaults, decorators, loops, extra functions or markdown.
Allowed: assignment, arithmetic, dict, subscripting, sum, Decimal, str, int,
Decimal.quantize, ROUND_HALF_UP, and generator expressions directly over lines
or usual. No product prices, preferences or case answers may be hardcoded.
Do not optimize baskets: the task agent chooses products. Keep code under 2500
characters. Explain the hypothesis briefly.\nDevelopment trajectories: '''+
    json.dumps(before_dev), TOOL_CHANGE)
totals_tool = compile_totals(tool_proposal['tool_code'])
for case in development:
    usual=list(case['usual'].items()); prices={k:v['price'] for k,v in REFERENCE.items()}
    result=totals_tool(usual,usual,prices)
    assert money(result['usual_item_subtotal'])==item_total(case['usual'])
    assert money(result['estimated_item_subtotal'])==item_total(case['usual'])
    assert money(result['estimated_item_savings'])==0
# Non-identical independent contract checks, using development products only.
result=totals_tool([('g42',1),('g32',1),('j16',1)],[('q42',1),('b16',2),('j16',1)],prices)
assert [money(result[k]) for k in ('estimated_item_subtotal','usual_item_subtotal','estimated_item_savings')]==[Decimal('9.37'),Decimal('12.26'),Decimal('2.89')]
print('Gemini-written tool:',tool_proposal['tool_code']);print('Creation usage:',creation_usage)

TOOL_ANSWER = json.loads(json.dumps(ANSWER))
TOOL_ANSWER['properties']['action']={'type':'string','enum':['answer','tool']}
PRICE_LINE={'type':'object','properties':{'listing_id':{'type':'string'},'price':{'type':'string'}},
            'required':['listing_id','price'],'additionalProperties':False}
TOOL_ANSWER['properties']['tool_prices']={'type':'array','items':PRICE_LINE}
TOOL_ANSWER['required']+=['action','tool_prices']

def comparison_run(job):
    level,use_tool,case,repeat=job
    row={'occasion':case['id'],'repeat':repeat,'request':case['preferences'],'tool_calls':0}
    contract=("You may call basket_totals to calculate the totals of your proposed basket. "
        "Return action='tool' and tool_prices containing the displayed price for each "
        "listing in your chosen AND usual baskets. Your lines are the chosen-basket "
        "arguments; usual-basket quantities are supplied by the host from the request. "
        "The host executes the tool and replaces the three money fields directly in "
        "the final answer. For action='tool', use '0' placeholders for these fields. "
        "Or action='answer' to calculate yourself with tool_prices empty." if use_tool else '')
    prompt=(BASELINE['instruction']+'\nProcedural memory: '+BASELINE['memory']+
        '\nTask: propose a cheaper acceptable refill or explain the next step. '
        'Use the captured ordinary displayed item prices, before unverified promotions, tax or fees. '
        'Return a full proposed basket (or the usual basket if keeping it), '
        'bare decimal dollar strings, source URLs, and a short explanation. '
        'A proposed basket is not a verified checkout.\n'+contract+
        '\nUsual basket by listing ID: '+json.dumps(case['usual'])+
        '\nShopper preferences: '+case['preferences']+'\nRetailer evidence: '+json.dumps(LISTINGS))
    try:
        answer,resources=ask(prompt,TOOL_ANSWER if use_tool else ANSWER)
        row.update(resources,answer=answer)
        if use_tool and answer['action']=='tool':
            row['tool_calls']=1
            lines=[(l['listing_id'],l['quantity']) for l in answer['lines']]
            if len(lines)>10 or any(type(q)is not int or not 1<=q<=10000 for k,q in lines):
                raise ValueError('Invalid bounded tool quantities')
            parsed={l['listing_id']:l['price'] for l in answer['tool_prices']}
            for k,q in lines+list(case['usual'].items()):
                if k not in REFERENCE or money(parsed[k])!=Decimal(REFERENCE[k]['price']):
                    raise ValueError('Wrong tool price argument')
            started=time.monotonic()
            result=totals_tool(lines,list(case['usual'].items()),parsed)
            row.update(tool_seconds=time.monotonic()-started,tool_result=result)
            answer.update(result)
        row['feedback']=grade(case,answer)
    except Exception as error:
        row.update(getattr(error,'resources',{}))
        row.update(request_error=type(error).__name__,request_status=getattr(error,'code',None),
            error_detail=str(error) if isinstance(error,(RuntimeError,ValueError,KeyError)) else None,
            feedback={'appropriate':False,'errors':['Request or execution failed; inspect separately']})
    return row

def comparison_evaluate(level,use_tool,cases):
    global THINKING_LEVEL
    THINKING_LEVEL=level
    with ThreadPoolExecutor(max_workers=2) as pool:
        return list(pool.map(comparison_run,[(level,use_tool,c,r) for c in cases for r in range(REPEATS)]))

def costs(rows):
    result=summary(rows)
    for field in ('prompt_tokens','output_tokens','thinking_tokens','tokens'):
        observed=[r[field] for r in rows if r.get(field) is not None]
        result[field]=sum(observed) if observed else None
        result[field+'_reported_attempts']=len(observed)
    result['tool_calls']=sum(r.get('tool_calls',0) for r in rows)
    result['unattributed_tokens'] = sum(r['tokens']-r['prompt_tokens']-r['output_tokens'] for r in rows if all(r.get(k) is not None for k in ('tokens','prompt_tokens','output_tokens')))
    return result

low_tool_dev=comparison_evaluate('LOW',True,development)
b,t=costs(before_dev),costs(low_tool_dev)
efficiency_retained=(not b['failed_requests'] and not t['failed_requests'] and
    t['appropriate']>=b['appropriate'] and t['tokens'] is not None and b['tokens'] is not None and t['tokens']<b['tokens'])
print('LOW tool development:',t,'| efficiency candidate retained:',efficiency_retained)
comparison={'LOW_no_tool':before_test}
for level,label,use_tool in [('MEDIUM','MEDIUM_no_tool',False),('LOW','LOW_tool_available',True),('MEDIUM','MEDIUM_tool_available',True)]:
    comparison[label]=comparison_evaluate(level,use_tool,held_out)
    print(label,costs(comparison[label]))
THINKING_LEVEL='LOW'
experiment.update(tool_ablation={'creation_proposal':tool_proposal,'creation_usage':creation_usage,
    'tool_development':low_tool_dev,'efficiency_retained_on_development':efficiency_retained,
    'held_out':comparison,'summary':{k:costs(v) for k,v in comparison.items()}},calls=calls,usage=usage)
print('All calls:',len(calls),'| reported tokens:',sum(u['tokens'] or 0 for u in usage))


Gemini-written tool: def basket_totals(lines, usual, prices):
    sub = sum((Decimal(prices[item[0]]) * item[1] for item in lines), Decimal("0"))
    u_sub = sum((Decimal(prices[item[0]]) * item[1] for item in usual), Decimal("0"))
    sav = u_sub - sub
    cents = Decimal("0.01")
    return {
        "estimated_item_subtotal": str(sub.quantize(cents, rounding=ROUND_HALF_UP)),
        "usual_item_subtotal": str(u_sub.quantize(cents, rounding=ROUND_HALF_UP)),
        "estimated_item_savings": str(sav.quantize(cents, rounding=ROUND_HALF_UP))
    }
Creation usage: {'tokens': 4270, 'prompt_tokens': 4022, 'output_tokens': 248, 'thinking_tokens': None, 'cached_tokens': None, 'seconds': 5.648}
LOW tool development: {'appropriate': 7, 'attempts': 8, 'occasions': 4, 'failed_requests': 0, 'tokens': 15064, 'incomplete_checklists': 7, 'prompt_tokens': 11104, 'prompt_tokens_reported_attempts': 8, 'output_tokens': 3960, 'output_tokens_reported_attempts': 8, 'thinking_tokens': None, 'thinking_tokens_

In [14]:
from IPython.display import display, JSON
display(JSON(experiment, expanded=False))


Expand the complete experiment, including failures, proposals and token records.